# Does the Route Matter? Base config and runs on Vertex AI + the Claude API

This notebook sets up the route-faithfulness harness (`route-faithfulness/` in this repo) and runs it on two models:

| Key in `config.yaml` | Model | Where it runs | Auth |
|---|---|---|---|
| `claude_sonnet46` | Claude Sonnet 4.6 (`claude-sonnet-4-6`) | Claude API | `ANTHROPIC_API_KEY` |
| `gemini35_flash` | Gemini 3.5 Flash (`google/gemini-3.5-flash`) | Vertex AI, OpenAI-compatible endpoint | gcloud ADC |
| judge | gpt-oss-120b (`openai/gpt-oss-120b-maas`) | Vertex AI | gcloud ADC |

Conditions (the plan's ladder, one change per step): **C1** Direct (D0) → **C2** Reshaped (D1) → **T1** Tool-injected (channel only) → **C3** Tool, model calls it (agency, T2), plus **C4** RAG-oracle and **C5** RAG-BM25.

**Where to run it**
- *Locally* (VS Code / Jupyter): open this file from the repo, use the project's `.venv` kernel, put keys in `route-faithfulness/.env` (see `.env.example`), and run `gcloud auth application-default login` once.
- *Colab Enterprise on Vertex AI*: Console → Vertex AI → Colab Enterprise → **Import** → URL → paste the raw GitHub URL of this notebook. Connect to a runtime (a small CPU machine such as e2-standard-2 is enough; the work is API calls). Set `GCS_URI` below, because the runtime's disk does not survive.

**Order of work:** run the cells top to bottom with `RUN_FULL = False`, check the smoke-test diagnostics, then set `RUN_FULL = True` and run the last section. Every step is resumable: rerunning a cell continues where it stopped and retries failed rows.

In [ ]:
# ---- Settings: edit these ----
PROJECT_ID = ""             # GCP project for Vertex AI. Empty = GOOGLE_CLOUD_PROJECT from .env, else the gcloud default
MODELS = ["claude_sonnet46", "gemini35_flash"]   # keys under `models:` in config.yaml
CONDITIONS = ["C1", "C2", "T1", "C3", "C4", "C5"]
SMOKE_PAIRS = 30            # smoke test: 30 pairs = 60 documents per condition, balanced across groups
RUN_FULL = False            # True = all 940 pairs. Only after the smoke-test checks below look right.
CONCURRENCY = 8             # parallel requests per model; lower it if you see rate limits
GCS_URI = ""                # e.g. "gs://my-bucket/route-faithfulness": outputs/ and results/ are synced there
REPO_URL = "https://github.com/sandeepstele/546"   # use https://github.com/dhanushya08/546 once the PR is merged
REPO_BRANCH = "vertex-claude-setup"                 # and "main" then

## 1. Get the code

In [ ]:
import os, subprocess, sys
from pathlib import Path

PY = sys.executable
IN_COLAB = "google.colab" in sys.modules


def sh(cmd, check=True):
    # Run a shell command and stream its output here (progress bars skipped).
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if "it/s]" not in line and "s/it]" not in line:
            print(line, end="", flush=True)
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")
    return rc


here = Path.cwd()
if (here / "src" / "run.py").exists():
    ROOT = here
elif (here.parent / "src" / "run.py").exists():       # opened from route-faithfulness/notebooks/
    ROOT = here.parent
else:                                                  # Colab Enterprise: clone the repo
    if not Path("546").exists():
        sh(f"git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} 546")
    ROOT = Path("546/route-faithfulness").resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Working in", ROOT)

if IN_COLAB:
    sh(f"{PY} -m pip install -q -r requirements.txt")
else:
    try:
        import anthropic, google.auth, openai, rank_bm25  # noqa: F401
    except ImportError:
        raise SystemExit("Missing packages: use the project's .venv kernel, or run "
                         "`pip install -r requirements.txt` in this kernel's environment.")

## 2. Credentials

- **Vertex AI** (Gemini and the judge) uses Google Application Default Credentials: on Colab Enterprise that is your own Google login; locally run `gcloud auth application-default login` once. API keys do not work on Vertex's OpenAI-compatible endpoint.
- **Claude API** needs `ANTHROPIC_API_KEY`. It is read from the environment or the git-ignored `.env`, then from Secret Manager (a secret named `anthropic-api-key`), and only then asked for. It is never printed or written into this notebook.

In [ ]:
import getpass
import google.auth
import google.auth.exceptions
from dotenv import load_dotenv

load_dotenv(ROOT / ".env")
from src.config import load_config

cfg = load_config()

try:
    _creds, adc_project = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
except google.auth.exceptions.DefaultCredentialsError:
    raise SystemExit("No Google credentials. Locally: `gcloud auth application-default login`.")
PROJECT_ID = PROJECT_ID or os.environ.get("GOOGLE_CLOUD_PROJECT") or adc_project
assert PROJECT_ID, "Set PROJECT_ID in the settings cell."
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
print("Vertex AI project:", PROJECT_ID)


def load_secret(env_name, secret_id):
    # Fill os.environ[env_name] without echoing the value. Returns where it came from.
    if os.environ.get(env_name):
        return "environment / .env"
    try:
        from google.cloud import secretmanager
        client = secretmanager.SecretManagerServiceClient()
        name = f"projects/{PROJECT_ID}/secrets/{secret_id}/versions/latest"
        os.environ[env_name] = client.access_secret_version(name=name).payload.data.decode().strip()
        return "Secret Manager"
    except Exception:
        pass
    os.environ[env_name] = getpass.getpass(f"{env_name}: ").strip()
    return "typed in"


providers = {cfg["models"][m].get("provider", "openai") for m in MODELS}
if "anthropic" in providers:
    print("ANTHROPIC_API_KEY from", load_secret("ANTHROPIC_API_KEY", "anthropic-api-key"))

## 3. One tiny call per model

Expect `OK` from every line. `reasoning=True` is expected only for the judge (gpt-oss always reasons); for the tested models it must be `False`, because every route is compared with thinking off.

In [ ]:
from src.llm import LLM


async def ping(name, mcfg):
    try:
        r = await LLM(mcfg).chat([{"role": "user", "content": "Reply with the single word OK."}], max_tokens=200)
        print(f"{name:18s} {mcfg['model']:28s} -> {r['content'][:30]!r:34s} reasoning={r['reasoning_present']}")
    except Exception as e:
        print(f"{name:18s} {mcfg['model']:28s} FAILED {type(e).__name__}: {str(e)[:200]}")


for m in MODELS:
    await ping(m, cfg["models"][m])
await ping("judge", cfg["judge"])

## 4. Data (FaithConflict, 940 pairs from Hugging Face)

In [ ]:
from src.config import read_jsonl, resolve

data_path = resolve(cfg["data"]["normalized_path"])
if not data_path.exists():
    sh(f"{PY} -m src.data --prepare")
docs = read_jsonl(data_path)
print(len(docs), "documents,", len({d['pair_id'] for d in docs}), "pairs")

## 5. Cloud Storage sync (skip locally; needed on Colab Enterprise)

In [ ]:
def sync(direction):
    if not GCS_URI:
        return
    for d in ("outputs", "results"):
        if direction == "down":
            sh(f"gcloud storage rsync --recursive {GCS_URI}/{d} {d}", check=False)
        elif Path(d).exists():
            sh(f"gcloud storage rsync --recursive {d} {GCS_URI}/{d}")


sync("down")   # pick up earlier runs, so this session resumes them

## 6. Smoke test

`SMOKE_PAIRS` pairs per model across all conditions, then the judge and the analysis. These rows are reused by the full run.

In [ ]:
conds = " ".join(CONDITIONS)
for m in MODELS:
    sh(f"{PY} -m src.run --model {m} --limit-pairs {SMOKE_PAIRS} --conditions {conds} --concurrency {CONCURRENCY}")
for m in MODELS:
    sh(f"{PY} -m src.judge --model {m} --conditions {conds}")
sh(f"{PY} -m src.analyze --models {' '.join(MODELS)}")
sync("up")

## 7. Check before the full run

The table should show: no errors, `reasoning` 0 for every condition, C3 `tool_called` close to 100%, C5 `recall` close to 100%, and `judge_invalid` 0. Then read a few transcripts below. The cost column extrapolates the smoke test's real token counts to all 1,880 documents per condition at list prices (check current prices before budgeting).

In [ ]:
import pandas as pd

PRICES = {  # USD per 1M tokens (input, output), list prices November 2026
    "claude-sonnet-4-6": (3.00, 15.00),
    "google/gemini-3.5-flash": (1.50, 9.00),
    "gemini-3.5-flash": (1.50, 9.00),
}
JUDGE_PER_ANSWER = (2000 * 0.09 + 400 * 0.36) / 1e6   # gpt-oss-120b on Vertex, rough tokens per judged answer
N_DOCS = 1880

rows = []
for m in MODELS:
    for c in CONDITIONS:
        raw = read_jsonl(f"outputs/raw/{m}/{c}.jsonl")
        judged = read_jsonl(f"outputs/judged/{m}/{c}.jsonl")
        if not raw:
            continue
        ok = [r for r in raw if not r.get("error")]
        pin, pout = PRICES.get(raw[0]["model_id"], (float("nan"), float("nan")))
        tok_in = sum(u["prompt_tokens"] or 0 for r in ok for u in r.get("usage") or [] if u)
        tok_out = sum(u["completion_tokens"] or 0 for r in ok for u in r.get("usage") or [] if u)
        per_doc = (tok_in * pin + tok_out * pout) / 1e6 / max(1, len(ok))
        rows.append({
            "model": m, "cond": c, "docs": len(raw), "errors": len(raw) - len(ok),
            "reasoning": sum(bool(r.get("reasoning_present")) for r in ok),
            "empty": sum(not r.get("answer") for r in ok),
            "tool_called": (sum(bool(r.get("tool_calls")) for r in ok) / len(ok)) if c in ("C3", "C6") else None,
            "recall": (sum(r.get("gold_rank") is not None for r in ok) / len(ok)) if c == "C5" else None,
            "judge_invalid": sum(j.get("code") is None for j in judged),
            "in_tok/doc": round(tok_in / max(1, len(ok))), "out_tok/doc": round(tok_out / max(1, len(ok))),
            "full_run_$": round(per_doc * N_DOCS, 2),
        })
diag = pd.DataFrame(rows)
display(diag)
for m, g in diag.groupby("model"):
    print(f"{m}: estimated full run ≈ ${g['full_run_$'].sum():.0f} model + "
          f"${JUDGE_PER_ANSWER * N_DOCS * len(g):.0f} judge")

In [ ]:
# Read transcripts: T1 (we wrote the tool call) and C3 (the model made it), opposing documents
for m in MODELS:
    for c in ("T1", "C3"):
        sh(f"{PY} -m src.peek --model {m} --condition {c} --polarity opposing --n 1")

In [ ]:
from IPython.display import Markdown

Markdown((ROOT / "results" / "summary.md").read_text(encoding="utf-8"))

## 8. Full run (all 940 pairs)

Runs both models in parallel (different providers, so they do not share rate limits), logging to `logs/<model>.log`, then judges and analyzes. If anything stops (rate limit, sleep, runtime restart), run this cell again: it resumes.

On Colab Enterprise a multi-hour run can outlive your browser session; for unattended runs, schedule this notebook as an execution (Colab Enterprise → Executions) with `RUN_FULL = True` and `GCS_URI` set.

In [ ]:
import time

if not RUN_FULL:
    print("RUN_FULL is False. Set it to True in the settings cell once the smoke-test checks look right.")
else:
    Path("logs").mkdir(exist_ok=True)
    procs = {}
    for m in MODELS:
        log = open(f"logs/{m}.log", "a")
        procs[m] = (subprocess.Popen(f"{PY} -m src.run --model {m} --conditions {conds} --concurrency {CONCURRENCY}",
                                     shell=True, stdout=log, stderr=subprocess.STDOUT), log)
    while any(p.poll() is None for p, _ in procs.values()):
        time.sleep(300)
        for m in MODELS:
            n = sum(len(read_jsonl(f"outputs/raw/{m}/{c}.jsonl")) for c in CONDITIONS)
            print(time.strftime("%H:%M"), f"{m}: {n}/{N_DOCS * len(CONDITIONS)} documents done", flush=True)
        sync("up")
    for m, (p, log) in procs.items():
        log.close()
        print(m, "exit code", p.returncode, "(nonzero: rerun this cell to resume)")
    for m in MODELS:
        sh(f"{PY} -m src.judge --model {m} --conditions {conds}")
    sh(f"{PY} -m src.analyze --models {' '.join(MODELS)}")
    sync("up")